# Process and trace Tellbreen data

In [2]:
%matplotlib qt

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.interpolate import interp1d
import os
from radar_niklas2 import ReadFile, SkyCalibration, Combine_channels
import pandas as pd


# Pick surface and bed

In [6]:
"""
stationary measurements (down) with snowpit data:

EXACT SNOWPIT
4_03_2026_1, 4_03_2026_2 [loc 1]
5_03_2026_1, 5_03_2026_2  [loc 2]
5_03_2026_6, 5_03_2026_7 [loc 3] 

9_03_2026_2 [near weather station]
"""

filename = "10_03_2026_2"
filename_sky_calibration = "09_03_2026_1"
max_range = 3.5 # meters

# direc = sorted(os.listdir("storage"))
from radar_niklas2 import ReadFile, SkyCalibration, Combine_channels
# for filename in direc:

Sky_calibrated = ReadFile(filename_sky_calibration)
data = SkyCalibration(Sky_calibrated, ReadFile(filename))
data = Combine_channels(data)

combined_power_log = data['combined_power_log'].T
x_data = data['x'] # Y axis
combined_power_log = combined_power_log[x_data<=max_range][:].T

x_data = x_data[x_data<max_range] # y axis
slowtime = data['slowtime']

max_value = np.max(combined_power_log, axis=1)
combined_power_log = np.transpose(combined_power_log)/max_value

# # make 2d array with 0 in lower 1/3, 1 in middle 1/3 and 2 in upper 1/3
# data = np.zeros((100, 100))
# data[0:33, :] = 0
# data[33:66, :] = 1
# data[66:100, :] = 2
print(np.shape(x_data))
print(np.shape(slowtime))

print(np.shape(combined_power_log))
plt.imshow(combined_power_log, aspect='auto')


('Bandwidth', 2500)
('Bandwidth', 2500)
(59,)
(228,)
(59, 228)


In [7]:
# x = np.arange(1,slowtime.shape[0])  # horizontal axis (distance)
# y = np.arange(1,x_data.shape[0])  # vertical axis (elevation)
x = slowtime
y = x_data


fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(
    combined_power_log,
    aspect="auto", 
    extent=[x.min(), x.max(), y.max(), y.min()] ,
    origin = "upper",
    clim=[0,1]
)
ax.set_title("Click guide points along the SURFACE, then press Enter")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Depth [m]")

plt.show()

surface_pts = plt.ginput(n=-1, timeout=0)
plt.close(fig)

surface_pts = np.asarray(surface_pts, dtype=float)
#print(surface_pts)

KeyboardInterrupt: 

In [ ]:
# bed picks
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(
    combined_power_log,
    aspect="auto",
    extent=[x.min(), x.max(), y.max(), y.min()] , 
    clim=[0,1]
)
ax.set_title("Click guide points along the BED, then press Enter")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Depth [m]")
plt.show()

bed_pts = plt.ginput(n=-1, timeout=0)
plt.close(fig)

bed_pts = np.asarray(bed_pts, dtype=float)
#print(bed_pts)

In [ ]:
# sort and interpolate
surface_pts = surface_pts[np.argsort(surface_pts[:, 0])]
bed_pts = bed_pts[np.argsort(bed_pts[:, 0])]

surface_f = interp1d(surface_pts[:, 0], surface_pts[:, 1],
                     bounds_error=False, fill_value="extrapolate")

bed_f = interp1d(bed_pts[:, 0], bed_pts[:, 1],
                 bounds_error=False, fill_value="extrapolate")

x_dense = np.linspace(x.min(), x.max(), combined_power_log.shape[1])

surface_interp = surface_f(x_dense)
bed_interp = bed_f(x_dense)

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
# plot picks on radargram

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(
    combined_power_log,
    aspect="auto",
    extent=[x.min(), x.max(), y.max(), y.min()] ,
    clim=[0,1]
)

# interpolated lines (smooth)
ax.plot(x_dense, surface_interp, 'r-', linewidth=2, label='Surface (interp)')
ax.plot(x_dense, bed_interp, 'b-', linewidth=2, label='Bed (interp)')

# overlay original picks
ax.plot(surface_pts[:, 0], surface_pts[:, 1], 'ro', alpha=0.5)
ax.plot(bed_pts[:, 0], bed_pts[:, 1], 'bo', alpha=0.5)
ax.set_xlabel("Time Elapsed [s]")
ax.set_ylabel("Depth (m)")

ax.set_title("Surface and bed picks (interpolated) - " + filename)
ax.legend()
plt.show()

In [ ]:
# plot ice thickness
snow_thickness = - surface_interp + bed_interp

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x_dense, snow_thickness, 'k-', linewidth=2)
ax.set_xlabel("Time Elapsed [s]")
ax.set_ylabel("Snow thickness (m)")
ax.set_title("Uncorrected Snow thickness - " + filename)
ax.grid(True, alpha=0.3)

mean_thickness = np.mean(snow_thickness)

ax.text(
    0.05, 0.95,
    f"Mean: {mean_thickness:.10f} m",
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
)

plt.show()

print(np.mean(snow_thickness))



"""
Saving


I want to save surface interp, bed interp, thickness interp, x-axis, combined_power_log
"""
# print(np.shape(surface_interp))
# print(np.shape(bed_interp))
# print(np.shape(snow_thickness))
# print(np.shape(x_dense))
# print(np.shape(combined_power_log))
# print(np.shape(slowtime))

# # matrix_to_save = np.array([surface_interp, bed_interp, snow_thickness, x_dense, combined_power_log, slowtime])
# df = pd.DataFrame({
#     'surface_interp': surface_interp,
#     'bed_interp': bed_interp,
#     'snow_thickness': snow_thickness,
#     'x_dense': x_dense,
#     # 'combined_power_log': combined_power_log,
#     'slowtime': slowtime
# })

# df2 = pd.DataFrame({
#     'combined_power_log': combined_power_log,
# })

# df.to_csv(filename + "pickdata.csv", index=False)
# df2.to_csv(filename + "power_log.csv", index=False)

folder = "PickData"
os.makedirs(folder, exist_ok=True)

np.savez(
    os.path.join(folder, filename + "_pickdata.npz"),
    surface_interp=surface_interp,
    bed_interp=bed_interp,
    snow_thickness=snow_thickness,
    x_dense=x_dense,
    combined_power_log=combined_power_log,
    slowtime=slowtime
)



df_1d = pd.DataFrame({
    "surface_interp": surface_interp,
    "bed_interp": bed_interp,
    "snow_thickness": snow_thickness,
    "x_dense": x_dense,
    "slowtime": slowtime
})

df_1d.to_csv(os.path.join(folder, filename + "_pick_vars.csv"), index=False)

df_2d = pd.DataFrame(combined_power_log)
df_2d.to_csv(os.path.join(folder, filename + "_combined_power_log.csv"), index=False)


-1.6373987301814317


# Extract bed power

This needs to be done with data before it is corrected for elevation. Also, it should be applied to data that has not undergone denoising or gain correction, since these steps change the amplitudes.

In [ ]:
# rd_pow = load.load_ramac.load_ramac(file)
# rd_pow.crop(0.13, dimension="twtt")
# #rd_pow.adaptivehfilt(25)
# rd_pow.vertical_band_pass(15, 100)
# rd_pow.data = rd_pow.data.astype(np.float64)
# rd_pow.restack(2)
# rd_pow.nmo(4)
# interp([rd_pow], spacing=1.)

In [ ]:
# # pull arrays from rd_pow
# radar_pow = np.ma.filled(rd_pow.data, np.nan).astype(np.float64)
# x_pow = np.asarray(rd_pow.dist)
# tt_us = np.asarray(rd_pow.travel_time)   # two-way travel time in microseconds
# tt_s = tt_us * 1e-6                      # convert to seconds

# # evaluate your interpolated picks on the rd_pow trace positions
# surface_on_pow = surface_f(x_pow)
# bed_on_pow = bed_f(x_pow)

# # ice thickness in meters
# ice_thickness = surface_on_pow - bed_on_pow

In [ ]:
# # convert ice thicknessto twtt to bed
# uice = 1.68e8   # m/s
# #bed_twtt_s = np.maximum(4 * ice_thickness / uice, 0)
# bed_twtt_s = 4 * ice_thickness / uice

In [ ]:
# bed_twtt_s

In [ ]:
# # extract peak bed power
# bed_half_window = 5   # +/- samples around expected bed
# n_samples, n_traces = radar_pow.shape

# bed_peak_idx = np.full(n_traces, np.nan)
# bed_peak_twtt_s = np.full(n_traces, np.nan)
# bed_amp = np.full(n_traces, np.nan)
# bed_power = np.full(n_traces, np.nan)

# for i in range(n_traces):
#     if np.isnan(bed_twtt_s[i]):
#         continue

#     it = np.argmin(np.abs(tt_s - bed_twtt_s[i]))

#     i0 = max(0, it - bed_half_window)
#     i1 = min(n_samples, it + bed_half_window + 1)

#     segment = radar_pow[i0:i1, i]

#     if segment.size == 0 or np.all(np.isnan(segment)):
#         continue

#     local_idx = np.nanargmax(np.abs(segment))
#     global_idx = i0 + local_idx

#     amp = np.abs(radar_pow[global_idx, i])

#     bed_peak_idx[i] = global_idx
#     bed_peak_twtt_s[i] = tt_s[global_idx]
#     bed_amp[i] = amp
#     bed_power[i] = amp**2

# bed_power_db = 10 * np.log10(np.maximum(bed_power, 1e-12)) # convert to db

In [ ]:
# vmax = np.percentile(np.abs(radar_pow), 98)

# fig, ax = plt.subplots(figsize=(14, 7))
# ax.imshow(
#     radar_pow,
#     aspect="auto",
#     cmap="gray",
#     extent=[x_pow.min(), x_pow.max(), tt_us.max(), tt_us.min()],
#     vmin=-vmax,
#     vmax=vmax
# )

# ax.plot(x_pow, bed_twtt_s * 1e6, 'b-', linewidth=2, label='Estimated bed TWTT')
# ax.scatter(x_pow, bed_peak_twtt_s * 1e6, s=10, c='red', zorder=5, label='Peak amp near bed')

# ax.set_xlabel("Distance (km)")
# ax.set_ylabel("TWTT (µs)")
# ax.set_title("Estimated bed TWTT and extracted bed peaks")
# ax.legend()
# plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(14, 5))
# ax.plot(x_pow, bed_power_db, 'k-', linewidth=1.5)
# ax.set_xlabel("Distance (km)")
# ax.set_ylabel("Bed return power (dB)")
# ax.set_title("Bed return power along profile")
# ax.grid(True, alpha=0.3)
# plt.show()

# Geometric spreading correction

In [ ]:
# # geometric spreading correction
# bed_power_corr = bed_power * (ice_thickness**4)

# bed_power_corr_db = 10 * np.log10(np.maximum(bed_power_corr, 1e-12))

In [ ]:
# # smooth
# smooth_power = gaussian_filter1d(bed_power_corr_db, sigma=10)

# # plot
# fig, ax = plt.subplots(figsize=(14, 5))

# ax.plot(x_pow, bed_power_corr_db, color='0.7', linewidth=1, label='Raw')
# ax.plot(x_pow, smooth_power, 'k-', linewidth=2, label='Smoothed')

# ax.set_xlabel("Distance (km)")
# ax.set_ylabel("Corrected bed power (dB)")
# ax.set_title("Bed power (geometric spreading corrected)")
# ax.grid(True, alpha=0.3)
# ax.legend()

# plt.show()

# Back of the envelope attenuation correction
We don't actually know the attenuation so we're just going to play with some values

In [ ]:
# alpha_db_per_m = 0.01   # attenuation rate (10 dB/km)
# bed_power_attn_corr_db = bed_power_corr_db + 2 * alpha_db_per_m * ice_thickness

In [ ]:
# smooth_attn = gaussian_filter1d(bed_power_attn_corr_db, sigma=10)

# fig, ax = plt.subplots(figsize=(14, 5))
# ax.plot(x_pow, bed_power_attn_corr_db, color='0.7', linewidth=1, label='Raw')
# ax.plot(x_pow, smooth_attn, 'k-', linewidth=2, label='Smoothed')

# ax.set_xlabel("Distance (km)")
# ax.set_ylabel("Bed reflectivity proxy (dB)")
# ax.set_title("Bed power (geometric + attenuation corrected)")
# ax.grid(True, alpha=0.3)
# ax.legend()

# plt.show()